# Bradford Bulls — Team Detection & Classification

## Approach: Robust Dominant Shirt Colour (Lab K-means)

Simple, proven approach from sports video analysis research:
1. **Detect** players with YOLO26n + ByteTrack
2. **Crop strictly** the chest/shirt area (top 15–40% of bbox, skip head & shorts)
3. **Remove grass-green** pixels from the crop
4. **Mean Lab colour** of remaining pixels = shirt fingerprint (3-dim only)
5. **K-means(k=3)** on shirt colours → Team A / Team B / Referee
6. **Majority-vote** per track ID → stable labels

No SigLIP. No pose keypoints. No complex models.
Lab colour is the actual discriminating feature for jerseys — simple and robust.

**Before running:** Runtime → Change runtime type → **T4 GPU**

## 1. Check GPU

In [ ]:
!nvidia-smi
import torch
print(f'CUDA : {torch.cuda.is_available()}  |  {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

## 2. Install

In [ ]:
!pip install -q ultralytics scikit-learn
print('Done')

## 3. Mount Drive + Configuration

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

# >>> UPDATE path <<<
VIDEO_PATH = '/content/drive/MyDrive/bradford_bulls/videos/M02_h264.mp4'

CONF_THRESH       = 0.50   # YOLO confidence
MIN_PLAYER_HEIGHT = 0.07   # min bbox height / frame height

# Shirt crop: fraction of bbox height
SHIRT_TOP    = 0.15   # skip head (top 15%)
SHIRT_BOTTOM = 0.40   # stop before shorts (bottom 60%)

WARMUP_FRAMES    = 50
SMOOTHING_WINDOW = 20

BOX_COLORS = {
    'Team A':  (255, 80,  80 ),
    'Team B':  (80,  80,  255),
    'Referee': (0,   220, 220),
}

stem       = Path(VIDEO_PATH).stem
OUTPUT_DIR = Path(f'/content/drive/MyDrive/bradford_bulls/team_detection/{stem}')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Video : {VIDEO_PATH}  |  exists: {Path(VIDEO_PATH).exists()}')
print(f'Output: {OUTPUT_DIR}')

## 4. Load YOLO26n

In [ ]:
from ultralytics import YOLO
yolo = YOLO('yolo26n.pt')
print('YOLO26n ready')

## 5. Helper Functions

In [ ]:
import cv2
import numpy as np


def get_shirt_color(frame_rgb, box):
    """
    Crop the shirt/chest region only (SHIRT_TOP – SHIRT_BOTTOM of bbox height),
    remove grass-green pixels, return median Lab colour [L, a, b].
    Returns None if too few non-green pixels remain.
    """
    x1, y1, x2, y2 = [int(v) for v in box]
    h = y2 - y1

    sy1 = y1 + int(SHIRT_TOP    * h)
    sy2 = y1 + int(SHIRT_BOTTOM * h)
    if sy2 <= sy1 + 4:
        return None

    crop = frame_rgb[sy1:sy2, x1:x2]
    if crop.size == 0:
        return None

    # Remove grass-green pixels
    hsv        = cv2.cvtColor(crop, cv2.COLOR_RGB2HSV)
    green_mask = (hsv[:, :, 0] >= 30) & (hsv[:, :, 0] <= 90) & (hsv[:, :, 1] > 40)
    lab        = cv2.cvtColor(crop, cv2.COLOR_RGB2LAB).astype(np.float32)
    px         = lab.reshape(-1, 3)
    keep       = ~green_mask.flatten()
    px         = px[keep] if keep.sum() >= 10 else px
    if len(px) == 0:
        return None
    return np.median(px, axis=0)   # [L, a, b]


def lab_to_rgb_display(lab_vec):
    """Convert a Lab triplet (OpenCV scale) to an 8-bit RGB tuple for display."""
    arr = np.array([[[int(lab_vec[0]), int(lab_vec[1]), int(lab_vec[2])]]], dtype=np.uint8)
    rgb = cv2.cvtColor(arr, cv2.COLOR_LAB2RGB)
    return tuple(rgb[0, 0].tolist())


def draw_label_box(img_bgr, box, label, color_bgr):
    x1, y1, x2, y2 = [int(v) for v in box]
    cv2.rectangle(img_bgr, (x1, y1), (x2, y2), color_bgr, 2)
    (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.50, 1)
    cv2.rectangle(img_bgr, (x1, y1 - th - 6), (x1 + tw + 6, y1), color_bgr, -1)
    cv2.putText(img_bgr, label, (x1 + 3, y1 - 4),
                cv2.FONT_HERSHEY_SIMPLEX, 0.50, (255, 255, 255), 1, cv2.LINE_AA)


print('Helper functions ready')

## 6. Warm-up — Collect Shirt Colours + K-means

Samples frames, extracts the **median Lab shirt colour** per player (3 numbers — no complex models),
then clusters into 3 groups.  Each cluster is shown as a **colour swatch + sample crops**.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.cluster import KMeans
from tqdm.notebook import tqdm

cap      = cv2.VideoCapture(VIDEO_PATH)
fps_v    = cap.get(cv2.CAP_PROP_FPS) or 30.0
total_f  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
H_frame  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
duration = total_f / fps_v
MIN_H_PX = int(MIN_PLAYER_HEIGHT * H_frame)
cap.release()
print(f'Video: {duration:.1f}s  ({total_f} frames @ {fps_v:.0f} fps)')

warmup_idx = np.linspace(0, total_f - 1, WARMUP_FRAMES, dtype=int)
all_colors = []   # (N, 3) Lab — one per player detection
all_crops  = []   # corresponding shirt crops for display

cap = cv2.VideoCapture(VIDEO_PATH)
for fidx in tqdm(warmup_idx, desc='Collecting shirt colours'):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(fidx))
    ok, frame = cap.read()
    if not ok:
        continue
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    for box in yolo(frame, verbose=False, conf=CONF_THRESH, classes=[0])[0].boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        if (y2 - y1) < MIN_H_PX:
            continue
        color = get_shirt_color(frame_rgb, (x1, y1, x2, y2))
        if color is not None:
            all_colors.append(color)
            # Save shirt crop for display
            h = y2 - y1
            sy1 = y1 + int(SHIRT_TOP * h)
            sy2 = y1 + int(SHIRT_BOTTOM * h)
            all_crops.append(frame_rgb[sy1:sy2, x1:x2].copy())
cap.release()

print(f'Collected {len(all_colors)} player shirt samples')

# ── K-means on 3-dim Lab colour vectors ───────────────────────────────────────
color_arr = np.array(all_colors)
kmeans    = KMeans(n_clusters=3, random_state=42, n_init=20)
kmeans.fit(color_arr)
labels    = kmeans.labels_
centers   = kmeans.cluster_centers_   # (3, 3) Lab

# ── Visualise: colour swatch + 4 sample crops per cluster ────────────────────
N_CROPS = 4
fig, axes = plt.subplots(3, N_CROPS + 1, figsize=(3*(N_CROPS+1), 9))

row_colors = ['#FF6600', '#3399FF', '#33CC33']

for cid in range(3):
    # Column 0: colour swatch
    rgb  = lab_to_rgb_display(centers[cid])
    cnt  = (labels == cid).sum()
    axes[cid][0].add_patch(plt.Rectangle((0,0), 1, 1, color=[v/255 for v in rgb]))
    axes[cid][0].set_xlim(0,1); axes[cid][0].set_ylim(0,1)
    axes[cid][0].set_title(f'CLUSTER {cid}\nRGB{rgb}\n{cnt} samples',
                            fontsize=9, fontweight='bold', color=row_colors[cid])
    axes[cid][0].axis('off')

    # Columns 1-N_CROPS: shirt crops closest to centroid
    idxs  = np.where(labels == cid)[0]
    dists = np.linalg.norm(color_arr[idxs] - centers[cid], axis=1)
    best  = idxs[np.argsort(dists)[:N_CROPS]]
    for j, idx in enumerate(best):
        axes[cid][j+1].imshow(all_crops[idx])
        axes[cid][j+1].axis('off')
    for j in range(len(best), N_CROPS):
        axes[cid][j+1].axis('off')

    fig.text(0.01, 0.83 - cid*0.33, f'CLUSTER {cid}',
             fontsize=12, fontweight='bold', color=row_colors[cid],
             va='center', rotation=90)

plt.suptitle('Shirt colour clusters (swatch + sample crops)\n'
             '→ Assign clusters in next cell',
             fontsize=12, fontweight='bold')
plt.tight_layout(rect=[0.04, 0, 1, 1])
plt.savefig(OUTPUT_DIR / 'shirt_clusters.png', dpi=130, bbox_inches='tight')
plt.show()
print('\nLook at each row: colour swatch (left) + sample shirt crops (right)')

## 7. Assign Cluster Labels  ← **UPDATE HERE**

Match each cluster number to Bradford / Opponent / Referee based on the swatches above.
- Bright / white swatch → the white-jersey team
- Dark swatch → the dark-jersey team
- Third → referee or ambiguous

In [ ]:
# >>> UPDATE based on the colour swatches above <<<
TEAM_A_CLUSTER   = 0
TEAM_B_CLUSTER   = 1
REFEREE_CLUSTER  = 2

TEAM_A_LABEL = 'Bradford'
TEAM_B_LABEL = 'Opponent'
REF_LABEL    = 'Referee'

CLUSTER_MAP = {TEAM_A_CLUSTER: 'Team A', TEAM_B_CLUSTER: 'Team B', REFEREE_CLUSTER: 'Referee'}
LABEL_MAP   = {'Team A': TEAM_A_LABEL, 'Team B': TEAM_B_LABEL, 'Referee': REF_LABEL}
CENTROIDS   = {CLUSTER_MAP[cid]: centers[cid] for cid in range(3)}


def classify_shirt(color_lab):
    """Nearest-centroid classification in Lab space."""
    best_team, best_dist = 'Team A', float('inf')
    for team, centroid in CENTROIDS.items():
        d = float(np.linalg.norm(color_lab - centroid))
        if d < best_dist:
            best_dist, best_team = d, team
    return best_team


print('Assignments:')
for cid in [TEAM_A_CLUSTER, TEAM_B_CLUSTER, REFEREE_CLUSTER]:
    rgb = lab_to_rgb_display(centers[cid])
    print(f'  Cluster {cid} → {LABEL_MAP[CLUSTER_MAP[cid]]:12s}  RGB{rgb}')

## 8. Generate Team Detection Video

In [ ]:
import subprocess
from collections import deque, Counter
from tqdm.notebook import tqdm

cap      = cv2.VideoCapture(VIDEO_PATH)
src_fps  = cap.get(cv2.CAP_PROP_FPS) or 30.0
total_f  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
W        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H_vid    = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
MIN_H_PX = int(MIN_PLAYER_HEIGHT * H_vid)

TMP_PATH      = '/tmp/team_detect_raw.mp4'
OUT_VIDEO     = OUTPUT_DIR / f'{stem}_team_detection.mp4'
writer        = cv2.VideoWriter(TMP_PATH, cv2.VideoWriter_fourcc(*'mp4v'), src_fps, (W, H_vid))
track_history = {}
timeline      = []

for fidx in tqdm(range(total_f), desc='Team detection'):
    ok, frame = cap.read()
    if not ok:
        break

    output    = frame.copy()
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    counts    = {'Team A': 0, 'Team B': 0, 'Referee': 0}

    results = yolo.track(frame, persist=True, verbose=False,
                         conf=CONF_THRESH, classes=[0],
                         tracker='bytetrack.yaml')

    if results[0].boxes.id is not None:
        for box, tid in zip(results[0].boxes,
                            results[0].boxes.id.int().tolist()):
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            if (y2 - y1) < MIN_H_PX:
                continue

            color = get_shirt_color(frame_rgb, (x1, y1, x2, y2))
            if color is None:
                continue

            raw_team = classify_shirt(color)

            if tid not in track_history:
                track_history[tid] = deque(maxlen=SMOOTHING_WINDOW)
            track_history[tid].append(raw_team)
            stable_team = Counter(track_history[tid]).most_common(1)[0][0]

            label     = LABEL_MAP.get(stable_team, stable_team)
            color_bgr = BOX_COLORS.get(stable_team, (180, 180, 180))
            draw_label_box(output, (x1, y1, x2, y2), label, color_bgr)
            if stable_team in counts:
                counts[stable_team] += 1

    writer.write(output)
    timeline.append({'frame': fidx, 'sec': round(fidx / src_fps, 2),
                     'Team A': counts['Team A'], 'Team B': counts['Team B'],
                     'Referee': counts['Referee']})

cap.release()
writer.release()

subprocess.run(
    ['ffmpeg', '-y', '-i', TMP_PATH,
     '-c:v', 'libx264', '-preset', 'fast', '-crf', '20', str(OUT_VIDEO)],
    check=True, capture_output=True
)
print(f'Video → {OUT_VIDEO}  ({OUT_VIDEO.stat().st_size/1e6:.1f} MB)')
print(f'Track IDs: {len(track_history)}')

## 9. Timeline Chart

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.DataFrame(timeline)
w  = max(1, int(src_fps))
for col in ['Team A', 'Team B', 'Referee']:
    df[f'{col}_s'] = df[col].rolling(w, min_periods=1).mean()

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
axes[0].plot(df['sec'], df['Team A_s'],  color='#5050FF', label=TEAM_A_LABEL, lw=1.5)
axes[0].plot(df['sec'], df['Team B_s'],  color='#FF5050', label=TEAM_B_LABEL, lw=1.5)
axes[0].plot(df['sec'], df['Referee_s'], color='#00CCCC', label=REF_LABEL,    lw=1.0, ls='--')
axes[0].set_ylabel('Players / frame'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[0].spines[['top', 'right']].set_visible(False)

tot = df['Team A_s'] + df['Team B_s'] + 1e-9
axes[1].stackplot(df['sec'],
    df['Team A_s']/tot*100, df['Team B_s']/tot*100,
    colors=['#5050FF','#FF5050'], alpha=0.6,
    labels=[f'{TEAM_A_LABEL} %', f'{TEAM_B_LABEL} %'])
axes[1].set_ylabel('Relative presence (%)'); axes[1].set_xlabel('Time (s)')
axes[1].set_ylim(0,100); axes[1].legend(loc='upper right'); axes[1].grid(alpha=0.3)
axes[1].spines[['top','right']].set_visible(False)

plt.suptitle(f'Team Presence — {stem}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'team_timeline.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Export CSV

In [ ]:
csv = OUTPUT_DIR / f'{stem}_team_timeline.csv'
df[['frame','sec','Team A','Team B','Referee']].to_csv(csv, index=False)
print(f'CSV → {csv}\n')
for team, label in LABEL_MAP.items():
    print(f'{label:12s}  avg {df[team].mean():.1f}/frame   max {int(df[team].max())}/frame')